In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# SMOTE for imbalance
from imblearn.over_sampling import SMOTE

In [2]:
# Load Dataset
df = pd.read_csv("../dataset/Telco_Customer_Churn.csv")
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
# Drop unnecessary column
df.drop("customerID", axis=1, inplace=True)

In [4]:
print(df.head())

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No  

In [5]:
#check total charge have null values
df["TotalCharges"].isnull().sum()

np.int64(0)

In [6]:
#check total charge have null Strings
(df["TotalCharges"] == " ").sum()

np.int64(11)

In [7]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

In [8]:
#check total charge have null values
df["TotalCharges"].isnull().sum()

np.int64(11)

In [9]:
# Fill missing values
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [10]:
#check monthly charges have null values
df["MonthlyCharges"].isnull().sum()

np.int64(0)

In [11]:
#check monthly charges have null Strings
(df["MonthlyCharges"] == " ").sum()

np.int64(0)

In [12]:
# Use One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [13]:
#save processed dataset
df.to_csv("../processed_datasets/knn_processed.csv", index=False)

In [14]:
print(df.head())

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0              0       1           29.85         29.85        False   
1              0      34           56.95       1889.50         True   
2              0       2           53.85        108.15         True   
3              0      45           42.30       1840.75         True   
4              0       2           70.70        151.65        False   

   Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0         True           False             False   
1        False           False              True   
2        False           False              True   
3        False           False             False   
4        False           False              True   

   MultipleLines_No phone service  MultipleLines_Yes  ...  StreamingTV_Yes  \
0                            True              False  ...            False   
1                           False              False  ...            False   
2                         

In [15]:
# Define Features and Target
X = df.drop("Churn_Yes", axis=1)
y = df["Churn_Yes"]

In [16]:
# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
#save the scaler
joblib.dump(scaler, "../trained_models/knn_scaler.pkl")
print("Scaler saved successfully!")

Scaler saved successfully!


In [18]:
#Handle Class Imbalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

In [19]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

In [20]:
#find best K value
best_k = 1
best_acc = 0

for k in range(1, 31):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    
    if acc > best_acc:
        best_acc = acc
        best_k = k

print("Best K:", best_k)
print("Best Accuracy:", best_acc)

Best K: 1
Best Accuracy: 0.8236714975845411


In [21]:
# Train Final Model (Optimized)
knn = KNeighborsClassifier(
    n_neighbors=best_k,
    weights='distance',
    metric='manhattan',
    n_jobs=-1
)

knn.fit(X_train, y_train)

,n_neighbors,1
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'manhattan'
,metric_params,None
,n_jobs,-1


In [22]:
# Do Predictions
y_pred = knn.predict(X_test)

In [23]:
# Evaluation
print("\nFinal Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))


Final Accuracy: 0.8352657004830918

Confusion Matrix:
 [[819 202]
 [139 910]]

Classification Report:
               precision    recall  f1-score   support

       False       0.85      0.80      0.83      1021
        True       0.82      0.87      0.84      1049

    accuracy                           0.84      2070
   macro avg       0.84      0.83      0.83      2070
weighted avg       0.84      0.84      0.84      2070



In [24]:
# Save trained model
joblib.dump(knn, "../trained_models/knn_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [25]:
knn = joblib.load("../trained_models/knn_model.pkl")

# Use directly
y_pred = knn.predict(X_test)